# Create dataframe of equipment calculations from sql dump

In [1]:
import sqlite3
import pandas as pd
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import re

In [2]:
# Create SQLite databases
conn = sqlite3.connect(config.SPARK_DATA / "2_Clean" / "spark_equipment_calculations.db")
cursor = conn.cursor()

In [3]:
# SQL dump file
sql_dump_file = config.SPARK_DATA / "1_Raw" / "equipment_tool_full_dump.sql"

with open(sql_dump_file, "r", encoding="utf-8") as f:
    sql_script = f.read()

In [4]:
# Clean the SQL script so that can be executed in SQLite

# Remove backticks
sql_script = sql_script.replace("`", "")

# Remove AUTO_INCREMENT, ENGINE, DEFAULT CHARSET, and COLLATE statements
sql_script = re.sub(r"AUTO_INCREMENT=\d+\s*", "", sql_script)
sql_script = re.sub(r"AUTO_INCREMENT\s*", "", sql_script)
sql_script = re.sub(r"ENGINE=\w+\s*", "", sql_script)
sql_script = re.sub(r"DEFAULT CHARSET=\w+\s*", "", sql_script)
sql_script = re.sub(r"COLLATE=\w+\s*", "", sql_script)

# Extract only the DROP TABLE, CREATE TABLE, and INSERT INTO statements
sql_script = re.findall(r"(DROP TABLE.*?;|CREATE TABLE.*?;|INSERT INTO.*?;)", sql_script, re.DOTALL)
sql_script = "\n".join(sql_script)

In [5]:
# Execute the cleaned SQL script to create tables and insert data
cursor.executescript(sql_script)
conn.commit()

In [13]:
# Import as dataframes
equipment_calculations = pd.read_sql("""
                                    SELECT *
                                    FROM equipment_info  \
                                    WHERE Equipment_Type IS NOT NULL
""", conn)
fc_calculations = pd.read_sql("""
                              SELECT * 
                              FROM fume_cupboard_info
""", conn)

In [14]:
# Save dataframes as csv files
equipment_calculations.to_csv(config.SPARK_DATA / "2_Clean" / "equipment_calculations.csv", index=False)
fc_calculations.to_csv(config.SPARK_DATA / "2_Clean" / "fc_calculations.csv", index=False)